In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "ujjivan_2.silver.bank_transaction_fraud_detection"

df = spark.table(SILVER_TABLE)

#----------------------------------
# Data Types
#----------------------------------

df = (
    df
    .withColumn("Transaction_Date",F.to_date("Transaction_Date","dd-MM-yyyy"))
    .withColumn("Transaction_Amount",F.col("Transaction_Amount").cast("double"))
    .withColumn("Account_Balance",F.col("Account_Balance").cast("double"))
    .withColumn("Age",F.col("Age").cast("int"))
    .withColumn("Is_Fraud",F.col("Is_Fraud").cast("int"))
)

#----------------------------------
# Time Features
#----------------------------------

df = (
    df
    .withColumn("Hour",F.hour("Transaction_Time"))
    .withColumn("DayOfWeek",F.dayofweek("Transaction_Date"))
    .withColumn("Month",F.month("Transaction_Date"))
    .withColumn("Weekend",
                F.when(F.col("DayOfWeek").isin(1,7),1).otherwise(0))
)

#----------------------------------
# High Value Flag
#----------------------------------

df = (
    df.withColumn(
        "HighValue",
        F.when(F.col("Transaction_Amount")>50000,1).otherwise(0)
    )
)

#----------------------------------
# Digital Transaction
#----------------------------------

digital_devices = [
    "Mobile Device",
    "POS Mobile App",
    "Voice Assistant",
    "Payment Gateway Device",
    "Virtual Card"
]

df = (
    df.withColumn(
        "DigitalTransaction",
        F.when(F.col("Transaction_Device").isin(digital_devices),1).otherwise(0)
    )
)

#----------------------------------
# Previous Transaction Amount
#----------------------------------

w = Window.partitionBy("Customer_ID").orderBy("Transaction_Date","Transaction_Time")

df = (
    df.withColumn(
        "Previous_Amount",
        F.lag("Transaction_Amount").over(w)
    )
)

df = (
    df.withColumn(
        "Amount_Difference",
        F.col("Transaction_Amount")-F.coalesce(F.col("Previous_Amount"),F.lit(0))
    )
)

#----------------------------------
# Rolling Average
#----------------------------------

rolling = (
    Window.partitionBy("Customer_ID")
    .orderBy("Transaction_Date","Transaction_Time")
    .rowsBetween(-10,-1)
)

df = (
    df.withColumn(
        "Avg_Last10",
        F.avg("Transaction_Amount").over(rolling)
    )
)

#----------------------------------
# Transaction Count
#----------------------------------

count_window = (
    Window.partitionBy("Customer_ID")
)

df = (
    df.withColumn(
        "Customer_Total_Transactions",
        F.count("*").over(count_window)
    )
)

#----------------------------------
# Merchant Diversity
#----------------------------------

merchant = (
    df.groupBy("Customer_ID")
      .agg(
          F.countDistinct("Merchant_Category")
          .alias("Merchant_Diversity")
      )
)

df = df.join(merchant,"Customer_ID")

#----------------------------------
# Write Feature Table
#----------------------------------

(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("ujjivan_2.gold.transaction_features")
)